# Word Embeddings

In this lab, we use [spaCy](https://spacy.io/)'s pre-trained word embeddings, then we train our own on a corpus of messages: a Skip-Gram with negative sampling, written in [PyTorch](https://pytorch.org/).

In [ ]:
import collections
import itertools
import pathlib
import re
import time

import numpy as np
import spacy
import torch

## Pre-trained Models With spaCy

To use a spaCy language model, you first need to download it (see [spaCy models](https://spacy.io/usage/models)). Not all of them contain word embeddings: for English, `en_core_web_sm` and `en_core_web_trf` have none.

We work with `en_core_web_lg` (382 MB), which maps 684,830 words to 342,918 vectors of 300 dimensions: the case variants of a word share one, so `the`, `The` and `THE` have the same vector.

In [ ]:
!python -m spacy download en_core_web_lg

## Usage

Once the model is loaded, `nlp` processes a text and returns a `Doc`, each token of which has its vector:

In [ ]:
# Load the model
nlp = spacy.load("en_core_web_lg")

# Process the text with spaCy
doc = nlp("This is some text that I am processing with spaCy")

print("Vector of the 4th token:")
print(doc[3].vector)

# The average vector of the sentence: one way to represent texts for
# classification, but a coarse one (word order is lost). Prefer a sentence
# embedding model (Sentence Transformers)
print("Average vector:")
print(doc.vector)

## A Table of Embeddings

The model's vectors are in `nlp.vocab.vectors`, a table with one row per word. The `Embeddings` class below holds them for the rest of the lab: `word in model` tells whether the model knows `word`, `model[word]` returns its vector, and `model.neighbors(vector)` the words closest to `vector`, as `(word, cosine)` pairs. It divides each row by its norm: the dot product of two rows is then their cosine, and `neighbors` compares a vector with every word in a single multiplication.

The case variants of a word share one row, which the table shows in lowercase: the neighbors of `Berlin` are written `berlin`, `hamburg`…

In [ ]:
class Embeddings:
  """A table of embeddings: one vector of length 1 per word."""

  def __init__(self, rows, table):
    self.rows = rows  # word -> number of its row in the table
    self.words = {}  # row number -> word (in lowercase if there are several)
    for word, row in rows.items():
      if row not in self.words or word.islower():
        self.words[row] = word
    # Rows of length 1: the dot product of two rows is their cosine
    self.table = table / np.linalg.norm(table, axis=1, keepdims=True)

  def __contains__(self, word):
    return word in self.rows

  def __getitem__(self, word):
    return self.table[self.rows[word]]

  def neighbors(self, vector, n=5):
    """The n words closest to `vector`: (word, cosine) pairs."""
    cosines = self.table @ (vector / np.linalg.norm(vector))
    return [
      (self.words[i], round(float(cosines[i]), 2)) for i in np.argsort(-cosines)[:n]
    ]


# spaCy's pre-trained vectors
vectors = nlp.vocab.vectors
model = Embeddings(
  {nlp.vocab.strings[key]: row for key, row in vectors.key2row.items()},
  vectors.data,
)
print(len(model.rows), "words,", model.table.shape)

In [ ]:
vector = model["easy"]
vector.shape

In [ ]:
model.neighbors(model["Berlin"])

In [ ]:
# The dot product of two vectors of length 1: their cosine similarity
print(model["Berlin"] @ model["Munich"])
print(model["Berlin"] @ model["Paris"])

## 1. Preprocessing Consistent With the Model's

`model` only knows the words of its vocabulary, written and split as in the texts it learned from: a missing word has no vector, and `model[word]` then raises `KeyError`.

Split `sentence` into words with `split`, then print the words the model knows, and those it doesn't. Do it again with the tokens of `nlp(sentence)`. Which splitting does the model expect? Finally, print the neighbors of `Berlin` and those of `berlin`: does case matter for this model? What does it tell you about preprocessing your own texts?

**Useful documentation**

- [`str.split`](https://docs.python.org/3/library/stdtypes.html#str.split)
- [`Token.text`](https://spacy.io/api/token#attributes)

In [ ]:
sentence = "Berlin and Munich are two of Germany's large cities"

# Your code here

### Additional hints

A list comprehension keeps the known words: `[word for word in words if word in model]`. The text of a token of `nlp(sentence)` is `token.text`. To check: `split` leaves `Germany's`, which the model doesn't know, and spaCy makes two tokens of it, `Germany` and `'s`.

```python
words = sentence.split()
print("Known:", [word for word in words if word in model])
print("Missing:", [word for word in words if ...])

words = [... for token in nlp(sentence)]
...

print(model.neighbors(model["Berlin"]))
...
```

### Solution

In [ ]:
words = sentence.split()
print("Known:", [word for word in words if word in model])
print("Missing:", [word for word in words if word not in model])

# spaCy's splitting, close to that of the texts these vectors learned from:
# Germany's becomes Germany and 's, which the model knows
words = [token.text for token in nlp(sentence)]
print("Known:", [word for word in words if word in model])
print("Missing:", [word for word in words if word not in model])

# Case doesn't matter here: Berlin and berlin share one vector, hence the same
# neighbors (other vectors keep case apart, as spaCy's French ones do)
print(model.neighbors(model["Berlin"]))
print(model.neighbors(model["berlin"]))

## 2. Analogies

The "Analogies" slide shows it: the offset from *man* to *woman* is the same as from *king* to *queen*, and king − man + woman ≈ queen. Hence a question: `x` is to `y` what `z` is to? The word sought is close to the vector `model[y] - model[x] + model[z]`.

First compute this vector for *France*, *Paris*, *Japan*, and print its nearest neighbors with `model.neighbors`. Which word comes first, and why? Then write `analogy(model, x, y, z)`: like `neighbors`, it returns the list of the `(word, cosine)` pairs closest to this vector, the best one first, but without the question's words. Try it on *France*, *Paris*, *Japan*, then on *man*, *woman*, *king*. The rest of the lab calls `analogy`.

In [ ]:
# Your code here: the offset and model.neighbors


def analogy(model, x, y, z):
  # Your code here
  ...

### Additional hints

`model["Paris"] - model["France"] + model["Japan"]` is a vector like any other: `model.neighbors(vector)` returns its 5 nearest neighbors, without leaving out any word. To leave out the question's words, compare rows: `model.rows[word]` is the row of `word` in the table, and `Japan` and `japan` share theirs. Ask for a few more neighbors (`n=8`), remove those whose row is that of a word of the question, and keep the first 5. For *France*, *Paris*, *Japan*, `tokyo` comes first in `analogy`.

```python
vector = model["Paris"] - ... + ...
print(model.neighbors(vector))


def analogy(model, x, y, z):
  vector = ...
  question = {model.rows[word] for word in (x, y, z)}
  closest = model.neighbors(vector, n=8)
  return [(word, c) for word, c in closest if ...][:5]


print(analogy(model, "France", "Paris", "Japan"))
print(analogy(model, ...))
```

### Solution

In [ ]:
vector = model["Paris"] - model["France"] + model["Japan"]
# japan first: the vector stays close to model["Japan"], and neighbors leaves
# out no word
print(model.neighbors(vector))


def analogy(model, x, y, z):
  # x is to y what z is to? The question's words are left out
  vector = model[y] - model[x] + model[z]
  # The rows of the question's words (Japan and japan share theirs)
  question = {model.rows[word] for word in (x, y, z)}
  closest = model.neighbors(vector, n=8)
  return [(word, c) for word, c in closest if model.rows[word] not in question][:5]


print(analogy(model, "France", "Paris", "Japan"))
# queen first
print(analogy(model, "man", "woman", "king"))

## 3. Successes, Failures and Biases

An analogy doesn't always work. Try `analogy` on *France*, *Paris*, *Italy*, then on *France*, *Paris*, *Australia*: which one gives the capital? What does *man*, *woman*, *doctor* give? The vectors come from their training corpus: what they associate, biases included, is what this corpus associates.

In [ ]:
# Your code here

### Additional hints

One call per triplet, in the statement's `x`, `y`, `z` order (*France* is to *Paris* what *Italy* is to?). For *France*, *Paris*, *Italy*, `rome` comes first.

```python
print(analogy(model, "France", "Paris", "Italy"))
print(analogy(model, ...))
print(analogy(model, ...))
```

### Solution

In [ ]:
# Right: rome
print(analogy(model, "France", "Paris", "Italy"))
# Wrong: sydney, melbourne, perth, and no canberra among the first five
print(analogy(model, "France", "Paris", "Australia"))
# nurse first: the corpus's associations
print(analogy(model, "man", "woman", "doctor"))

## Training Word Embeddings

Learning vectors on your own data isn't complicated. It's particularly useful on a specialized corpus (notarial deeds, ancient languages…), whose words don't mean the same elsewhere, or don't exist elsewhere.

These static embeddings, one vector per word, used to be trained with a dedicated library, such as gensim or fastText: that's the historical way. Today, word representations are learned inside the models that use them (transformers), and gensim is [in maintenance mode](https://github.com/piskvorky/gensim), with no new features. So we write the Skip-Gram ourselves, in PyTorch, in about fifty lines.

### The 20 Newsgroups Corpus

This corpus gathers about 20,000 messages in English, split into 20 themes, since they come from 20 different newsgroups.

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
!tar xzf 20news-bydate.tar.gz
!ls 20news-bydate-train/

In [ ]:
!ls 20news-bydate-test/

In [ ]:
# One folder per newsgroup, one message per file
contents = [
  p.read_text(encoding="latin-1") for p in pathlib.Path(".").glob("20news-bydate-*/*/*")
]
print(f"{len(contents)} texts retrieved")

In [ ]:
print(contents[1])

In [ ]:
# Remove the message's header, the lines introducing a quote ("… writes:")
# and the punctuation, then split into words
def preprocess(content: str) -> list[str]:
  start = content.find("\n\n")  # The header ends at the first blank line
  if start > 0:
    content = content[start + 2 :]
  words_per_line = (
    re.sub(r'[\!"#$%&\*+,-./:;<=>?@^_`()|~=]', " ", line).split()
    for line in content.splitlines()
    if not line.endswith("writes:")
  )
  return list(itertools.chain.from_iterable(words_per_line))


texts = [preprocess(content) for content in contents]

In [ ]:
print(texts[0])

## Detecting Frequent Expressions (Collocations)

Some word pairs are very common (*New York*): they're better treated as a single "word" of the vocabulary, `New_York`.

The cell below keeps the pairs of words that follow each other far more often than chance would predict, with the score of the word2vec paper ([Mikolov et al., 2013](https://arxiv.org/abs/1310.4546)): $\frac{\text{count}(ab) - \delta}{\text{count}(a) \times \text{count}(b)}$, where $\delta$ leaves out the pairs that are too rare; multiplied by the number of words in the corpus, this score says how many times more often than chance the pair occurs. With no grammatical analysis, it finds proper names (`New_York`, `United_States`), but also uninteresting pairs such as `at_least`. It takes about twenty seconds.

In [ ]:
# Two words that follow each other far more often than chance would have it make
# an expression: score of Mikolov et al. (2013), (count(ab) - δ) / (count(a) × count(b))
word_counts = collections.Counter(word for text in texts for word in text)
pair_counts = collections.Counter(p for text in texts for p in zip(text, text[1:]))
total = sum(word_counts.values())  # × total: the score becomes a ratio to chance
phrases = {
  (a, b)
  for (a, b), count in pair_counts.items()
  if (count - 5) * total / (word_counts[a] * word_counts[b]) > 100
}


def group(text):
  # Replace each pair of an expression with a single word, a_b
  grouped = []
  for word in text:
    if grouped and (grouped[-1], word) in phrases:
      grouped[-1] = grouped[-1] + "_" + word
    else:
      grouped.append(word)
  return grouped


grouped_texts = [group(text) for text in texts]
print(grouped_texts[0])

## 4. Training a Skip-Gram

Skip-Gram predicts the neighbors of a word from the central word (the "Learning (CBOW and Skip-Gram)" slide). It learns two vectors per word: that of the word at the center of a window, and that of the word as a neighbor; the score of a (center, neighbor) pair is the dot product of these two vectors. Rather than predicting the neighbor among all the words of the vocabulary, negative sampling learns to tell the true neighbor from a few words drawn at random, the *negatives*.

The cell below prepares the data: the vocabulary (the words seen at least 5 times, from the most frequent to the rarest), the corpus as a single sequence of word numbers, and a subsampling, which removes the very frequent words (*the*, *of*…) most of the time. `torch.manual_seed(0)` fixes the starting point of the randomness, for the subsampling as for the training that follows.

In [ ]:
# Vocabulary: the words seen at least 5 times, from the most frequent to the rarest
counts = collections.Counter(word for text in grouped_texts for word in text)
vocabulary = [word for word, count in counts.most_common() if count >= 5]
numbers = {word: number for number, word in enumerate(vocabulary)}
frequencies = torch.tensor([counts[word] for word in vocabulary], dtype=torch.float)
# The corpus: a single sequence of word numbers
corpus = torch.tensor(
  [numbers[word] for text in grouped_texts for word in text if word in numbers]
)
# Subsampling: a very frequent word (the, of…) is removed most of the time
torch.manual_seed(0)
keep = torch.rand(len(corpus)) < torch.sqrt(
  1e-3 * frequencies.sum() / frequencies[corpus]
)
corpus = corpus[keep]
print(len(vocabulary), "words,", len(corpus), "occurrences kept")

### 4.1. The Model and Its Loss

Complete the `forward` method of `SkipGram` and the `loss` function. `SkipGram` has two tables of vectors, `centers` and `contexts`, already created: `forward(center, context)` returns the score of each pair, the dot product of the vector of `center` in `centers` and that of `context` in `contexts`. `loss(skipgram, center, context, negatives)` receives a batch of centers and of true neighbors (word numbers, each of shape `(batch,)`) and, for each center, `k` negatives (of shape `(batch, k)`). It returns the mean over the batch of the negative sampling loss, for a center $c$, its true neighbor $o$ and its negatives $n_1, \dots, n_k$:

$$-\log \sigma(s(c, o)) - \sum_{i=1}^{k} \log \sigma(-s(c, n_i))$$

where $s$ is the score and $\sigma$ the sigmoid. It is low when the true neighbor has a high score and the negatives a low one.

**Useful documentation**

- [`torch.nn.Embedding`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)
- [`torch.nn.functional.logsigmoid`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.logsigmoid.html)
- [`torch.Tensor.sum`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.sum.html)
- [`torch.Tensor.mean`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.mean.html)
- [Broadcasting](https://docs.pytorch.org/docs/stable/notes/broadcasting.html)

In [ ]:
class SkipGram(torch.nn.Module):
  def __init__(self, vocabulary_size, dimension):
    super().__init__()
    # Two tables: the vector of a word at the center, and its vector as a neighbor
    self.centers = torch.nn.Embedding(vocabulary_size, dimension, sparse=True)
    self.contexts = torch.nn.Embedding(vocabulary_size, dimension, sparse=True)
    # word2vec's initialization: small centers, zero contexts
    torch.nn.init.uniform_(self.centers.weight, -0.5 / dimension, 0.5 / dimension)
    torch.nn.init.zeros_(self.contexts.weight)

  def forward(self, center, context):
    # Your code here
    ...


def loss(skipgram, center, context, negatives):
  # Your code here
  ...

#### Additional hints

`self.centers(center)` returns the vectors of the words of `center`, of shape `(batch, dimension)`; the elementwise product `*`, then `.sum(dim=-1)` over the last dimension, gives the dot products. In `loss`, `skipgram(center, context)` returns the scores of the true pairs, and `skipgram(center[:, None], negatives)` those of the negatives, of shape `(batch, k)`: `center[:, None]`, of shape `(batch, 1)`, broadcasts against `negatives`. `torch.nn.functional.logsigmoid(x)` computes $\log \sigma(x)$; the sum over the negatives is over `dim=1`, the mean over the whole batch. To check: with zero contexts, as at initialization, every score is 0, and the loss is $(k + 1) \log 2$, that is 4.159 for 5 negatives.

```python
class SkipGram(torch.nn.Module):
  # __init__ as in the exercise cell

  def forward(self, center, context):
    return (self.centers(center) * ...).sum(dim=...)


def loss(skipgram, center, context, negatives):
  real = torch.nn.functional.logsigmoid(skipgram(center, context))
  noise = torch.nn.functional.logsigmoid(...).sum(dim=1)
  return ...


negatives = torch.tensor([[2, 3, 4, 5, 6]])
print(loss(SkipGram(10, 4), torch.tensor([0]), torch.tensor([1]), negatives))  # 4.159
```

#### Solution

In [ ]:
class SkipGram(torch.nn.Module):
  def __init__(self, vocabulary_size, dimension):
    super().__init__()
    # Two tables: the vector of a word at the center, and its vector as a neighbor
    self.centers = torch.nn.Embedding(vocabulary_size, dimension, sparse=True)
    self.contexts = torch.nn.Embedding(vocabulary_size, dimension, sparse=True)
    # word2vec's initialization: small centers, zero contexts
    torch.nn.init.uniform_(self.centers.weight, -0.5 / dimension, 0.5 / dimension)
    torch.nn.init.zeros_(self.contexts.weight)

  def forward(self, center, context):
    # Score of a pair: the dot product of its two vectors
    return (self.centers(center) * self.contexts(context)).sum(dim=-1)


def loss(skipgram, center, context, negatives):
  # -log σ(score of the true neighbor) - Σ log σ(-score of a negative), averaged over the batch
  real = torch.nn.functional.logsigmoid(skipgram(center, context))
  noise = torch.nn.functional.logsigmoid(-skipgram(center[:, None], negatives)).sum(
    dim=1
  )
  return -(real + noise).mean()

### 4.2. Training

The loop below trains the Skip-Gram on `corpus`. At each step, it takes a batch of 4,096 positions of the corpus, a random neighbor in the window of each, and 5 negatives per center, drawn according to the frequency of the words raised to the power 0.75; `SparseAdam` only updates the rows of the batch's words. The three epochs take three to six minutes on two cores, depending on their load: run it, and read on while it trains. The printed loss goes down from one epoch to the next, from about 2.8 to 2.4.

The learned vectors, those of the `centers` table, then go into an `Embeddings`, like spaCy's: `model_forums`.

In [ ]:
DIMENSION, WINDOW, NEGATIVES, BATCH = 100, 5, 5, 4096
EPOCHS = 3

skipgram = SkipGram(len(vocabulary), DIMENSION)
optimizer = torch.optim.SparseAdam(skipgram.parameters(), lr=0.003)
negative_weights = frequencies**0.75  # a little more chance for rare words

start = time.time()
for epoch in range(EPOCHS):
  total = 0.0
  batches = torch.randperm(len(corpus) - 2 * WINDOW).add(WINDOW).split(BATCH)
  for positions in batches:
    # A neighbor drawn in the window: 1..WINDOW words away, to the left or right
    offset = torch.randint(1, WINDOW + 1, positions.shape)
    offset = offset * (2 * torch.randint(0, 2, positions.shape) - 1)
    center, context = corpus[positions], corpus[positions + offset]
    negatives = torch.multinomial(
      negative_weights, len(positions) * NEGATIVES, replacement=True
    )
    value = loss(skipgram, center, context, negatives.view(-1, NEGATIVES))
    optimizer.zero_grad()
    value.backward()
    optimizer.step()
    total += value.item()
  print(
    f"epoch {epoch + 1}: loss {total / len(batches):.3f}, {time.time() - start:.0f} s"
  )

# The learned vectors, in the same class as spaCy's
model_forums = Embeddings(numbers, skipgram.centers.weight.detach().numpy())

### 4.3. What a Corpus of the Domain Brings

Compare the neighbors of `Windows`, `DOS`, `dos` and `New_York` in `model_forums` and in `model`, spaCy's vectors: what does a corpus of your domain bring? Do both models know all these words? Finally, try `analogy(model_forums, "man", "woman", "king")`: why does this small corpus fail, where spaCy's vectors found *queen* (exercise 2)?

In [ ]:
# Your code here

#### Additional hints

`model_forums` is an `Embeddings`, like `model`: same `neighbors`, same `in`, and `analogy` works on it. `New_York` only exists in `model_forums`: test `word in model` before `model[word]`. In `model_forums`, `Windows_NT` is among the neighbors of `Windows`, and those of `DOS` and `dos` are computing words; in `model`, `DOS` and `dos` share one vector, whose neighbors are `os`, `todos`, `que`: Spanish.

```python
for word in ["Windows", "DOS", "dos", "New_York"]:
  print("20 Newsgroups", word, model_forums.neighbors(model_forums[word]))
  if word in model:
    print("spaCy", word, ...)
  else:
    ...

print(analogy(...))
```

#### Solution

In [ ]:
# 20 Newsgroups: Windows, DOS and dos have the computing of 1993 as neighbors
# (Windows_NT, Win_3…), and New_York less clear ones, which change with the
# seed; spaCy: today's Windows, and a DOS that is the Spanish dos
for word in ["Windows", "DOS", "dos", "New_York"]:
  print("20 Newsgroups", word, model_forums.neighbors(model_forums[word]))
  if word in model:
    print("spaCy", word, model.neighbors(model[word]))
  else:
    print("spaCy", word, "missing")

# Fewer than 5 million words, few kings and queens: the analogy fails
print(analogy(model_forums, "man", "woman", "king"))